# Laboratorio — Autocompletar: modelo que sugiere la siguiente palabra

**Universidad del Valle**
**Deep Learning y Sistemas Inteligentes — 2026**
**Kevin Recinos**

---

### Pregunta guía

Los teclados de dispositivos móviles sugieren tres palabras mientras el usuario escribe.
**¿Con qué precisión predice un modelo la siguiente palabra, cuántas pulsaciones ahorra efectivamente y en qué condiciones una red neuronal supera a un modelo basado en conteos?**
Usted construirá el modelo, lo medirá y lo someterá a prueba.

### Cómo se trabaja

| Bloque | Qué hará | Puntos |
|:--|:--|--:|
| 0 | **Investigar.** Dos lecturas y tres preguntas. | 10 |
| 1 | **Desarrollar.** Corpus, vocabulario y partición. | 5 |
| 2 | **Desarrollar.** Modelo de n-gramas de palabras (NumPy / Python). | 10 |
| 3 | **Desarrollar.** Sugerencias con retroceso (*backoff*). | 15 |
| 4 | **Desarrollar.** Medir como un usuario: pulsaciones ahorradas. | 15 |
| 5 | **Desarrollar.** Una red neuronal que predice la siguiente palabra (PyTorch). | 10 |
| 6 | **Investigar.** ¿Quién gana y en qué? Tabla y experimento propio. | 30 |
| 7 | Análisis y conclusión. | 5 |

**Requisitos:** conexión a internet una sola vez, para descargar el corpus (2 MB). El entrenamiento de la red toma menos de 1 minuto en CPU.

### Política de uso de inteligencia artificial

Puede utilizar asistentes de IA como apoyo para programar o investigar. Debe poder explicar cada función, métrica y decisión incluida en su notebook. La IA no constituye una fuente académica: verifique los resultados y las referencias. Una respuesta que no pueda defender no recibe el punto correspondiente, y el docente puede solicitarle una explicación oral de su trabajo. Presentar como propio un trabajo que no comprende o que no elaboró constituye plagio y se sanciona conforme al Reglamento de Evaluación de los Aprendizajes de la universidad.

### Vocabulario mínimo

| Término | Significa |
|:--|:--|
| **Token / vocabulario** | Cada palabra del texto; el vocabulario son las 5,000 más frecuentes. El resto se marca `<unk>`. |
| **n-grama** | Cuenta, para cada contexto de $n$ palabras, qué palabra le sigue. |
| **Retroceso (*backoff*)** | Si el contexto largo nunca se vio, usar uno más corto: trigrama → bigrama → unigrama. |
| **Top-$k$** | Acierto si la palabra real está entre las $k$ sugeridas. |
| **Pulsaciones ahorradas** | $1-\frac{\text{pulsaciones con sugerencias}}{\text{letras de las palabras}}$. Es lo que el usuario siente. |
| **Embedding** | Vector aprendido que representa una palabra; permite generalizar a contextos nunca vistos. |

In [1]:
import hashlib, itertools, math, os, re, urllib.request
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)
print("NumPy:", np.__version__, "| PyTorch:", torch.__version__)

NumPy: 2.4.2 | PyTorch: 2.13.0+cpu


---
## Bloque 0 — Investigación previa (10 puntos)

Lea, en este orden:

- **Jurafsky y Martin**, *Speech and Language Processing*, 3.ª ed., capítulo 3 "N-gram Language Models" (secciones 3.1 y 3.2, y la idea de retroceso):
  https://web.stanford.edu/~jurafsky/slp3/3.pdf
- **Bengio et al. (2003)**, *A Neural Probabilistic Language Model* (solo resumen e introducción):
  https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf

Responda con sus propias palabras, 2 a 3 líneas cada una.

1. Un trigrama predice la siguiente palabra mirando solo las dos anteriores. ¿Qué información útil queda fuera y qué pasa cuando el contexto de dos palabras nunca apareció en el texto de entrenamiento?
2. Los teclados móviles ofrecen **tres** sugerencias y no una. ¿Qué gana el usuario con eso y qué métrica refleja mejor esa decisión: top-1 o top-3?
3. Según Bengio, ¿qué problema de los n-gramas ("la maldición de la dimensionalidad") resuelven los *embeddings* y por qué permite generalizar a frases nunca vistas?

### Sus respuestas — Bloque 0

1. Se pierde información anterior, como el tema de la conversación o una referencia mencionada varias palabras antes. Si las dos palabras del contexto nunca aparecieron juntas, no se puede estimar su continuación por conteo directo; se recurre a un contexto más corto.

2. Se ofrecen varias alternativas y aumenta la posibilidad de encontrar la palabra deseada sin escribirla completa. Top-3 refleja mejor esa decisión porque cuenta el acierto dentro de las tres sugerencias, aunque el ahorro de pulsaciones mide mejor su utilidad práctica.

3. Al aumentar el vocabulario y el contexto, existen demasiadas combinaciones y muchas nunca aparecen en el entrenamiento. Los embeddings permiten compartir información entre palabras con usos parecidos, de modo que se pueden estimar continuaciones para combinaciones nuevas.

Referencias: Jurafsky y Martin, Speech and Language Processing, capítulo 3, https://web.stanford.edu/~jurafsky/slp3/3.pdf. Bengio et al. (2003), A Neural Probabilistic Language Model, https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf.


---
## Bloque 1 — Corpus y vocabulario (5 puntos)

El corpus es *Don Quijote de la Mancha* (Project Gutenberg, dominio público): unas 380,000 palabras de español.
El texto es literario y su registro difiere del de los mensajes actuales; se utiliza por ser de dominio público y de tamaño suficiente para estudiar el problema con texto real. Se utilizará el 90 % para entrenar y el 10 % final para validar.

> Si no tiene internet, descargue el archivo desde la URL de la celda y guárdelo como `quijote.txt` junto a este notebook.

In [2]:
URL = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"
if not os.path.exists("quijote.txt"):
    urllib.request.urlretrieve(URL, "quijote.txt")

crudo = open("quijote.txt", encoding="utf-8").read()
crudo = crudo[crudo.index("*** START"):crudo.index("*** END")]
crudo = crudo[crudo.index("\n") + 1:]
tokens = re.findall(r"[a-záéíóúüñ]+", crudo.lower())
print("palabras en el corpus:", len(tokens), "| distintas:", len(set(tokens)))
print(tokens[200:215])

palabras en el corpus: 383637 | distintas: 22944
['del', 'peligro', 'en', 'que', 'se', 'vio', 'con', 'una', 'turba', 'de', 'yangüeses', 'de', 'lo', 'que', 'le']


Construya el vocabulario de las 4,999 palabras más frecuentes más el símbolo `<unk>` en la posición 0 (total 5,000), el diccionario palabra → índice y la lista de índices del corpus (las palabras fuera del vocabulario valen 0).

> Pista: `Counter(tokens).most_common(4999)`; use `a_id.get(w, 0)`.

In [3]:
vocab = ["<unk>"] + [w for w, _ in Counter(tokens).most_common(4999)]
a_id = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

ids = [a_id.get(w, 0) for w in tokens]

corte = int(len(ids) * 0.9)
ent, val = ids[:corte], ids[corte:]
print("vocabulario:", V, "| entrenamiento:", len(ent), "| validación:", len(val))
print("fracción <unk>:", round(sum(i == 0 for i in ids) / len(ids), 4))

vocabulario: 5000 | entrenamiento: 345273 | validación: 38364
fracción <unk>: 0.0775


In [4]:
# ---- Verificación automática por hash (no modifique esta celda) ----
_HASHES = {
    "vocab": "41e2d3388885b32cb6091463a7e7c4a7973431d56b214361d2cb02949fccea46",
    "ngramas": "ae2f556b454c92532950627ff8c6c67eec9de684ce22fe0340ed31a0f64984c2",
    "ranking": "67d4d12e8443f788ee60544047c7722096a5c9efe56c4f1505f84857d931def6",
    "ahorro": "55c7a001c15888e0da202d2a0da258edd8c61e49c1f04e8b481288db8d7054f6",
    "n_params": "570189488c86b999aa2801ce9ef67e720a75efe2d02e67d62fe4f07c71350429",
    "red_ok": "6c3c396ed6b5c36dcae172271f462051b1266b851e92df3deea8ac65478fd712"
}


def _hash(v, dec=4):
    return hashlib.sha256(np.round(np.array(v, dtype=float), dec).tobytes()).hexdigest()


def verificar(clave, valor):
    ok = _hash(valor) == _HASHES[clave]
    print(("OK      " if ok else "REVISAR "), clave)
    return ok

In [5]:
verificar("vocab", [V, a_id["que"], a_id["quijote"], len(ids), sum(i == 0 for i in ids)]);

OK       vocab


---
## Bloque 2 — n-gramas de palabras (10 puntos)

Para cada contexto de $n$ palabras, contar qué palabra le sigue:

$$P(w_t \mid w_{t-n},\dots,w_{t-1}) \;=\; \frac{\text{cuenta}(\text{contexto}, w_t)}{\text{cuenta}(\text{contexto})}$$

Para $n=0$ el contexto es la tupla vacía `()` y se obtiene el modelo de unigramas (las palabras más frecuentes).

In [6]:
def entrenar(ids, n):
    # Devuelve un dict: contexto (tupla de n índices) -> Counter({siguiente_índice: veces}).
    tabla = defaultdict(Counter)
    for i in range(len(ids) - n):
        tabla[tuple(ids[i:i + n])][ids[i + n]] += 1
    return tabla


T = {n: entrenar(ent, n) for n in (0, 1, 2, 3)}
for n in T:
    print(f"n={n} | contextos distintos: {len(T[n]):>7}")

n=0 | contextos distintos:       1
n=1 | contextos distintos:    4991
n=2 | contextos distintos:   98919
n=3 | contextos distintos:  235019


In [7]:
ctx = (a_id["don"],)
print("después de 'don':", [(vocab[w], c) for w, c in T[1][ctx].most_common(5)])
verificar("ngramas", [len(T[1]), len(T[2]), len(T[3]), T[1][ctx][a_id["quijote"]], sum(T[0][()].values())]);

después de 'don': [('quijote', 1863), ('fernando', 131), ('luis', 36), ('diego', 27), ('lorenzo', 21)]
OK       ngramas


**Pregunta de análisis:** compare el número de contextos distintos de $n=1,2,3$ con el tamaño del corpus. (a) ¿Cuántas veces, en promedio, se vio cada contexto de trigrama? (b) ¿Qué consecuencia tiene eso al predecir sobre texto nuevo?

Se obtuvieron 4,991 contextos de una palabra, 98,919 de dos y 235,019 de tres, frente a 345,273 palabras de entrenamiento. Para el trigrama convencional, que usa dos palabras previas, cada contexto se observó unas 3.49 veces en promedio; con tres palabras previas, el promedio baja a 1.47.
Esto indica que muchos contextos tienen poca evidencia. En texto nuevo aparecen combinaciones desconocidas y los conteos resultan poco confiables, por lo que se necesita retroceso. En este notebook, n representa la cantidad de palabras del contexto.


---
## Bloque 3 — Sugerencias con retroceso (15 puntos)

Un trigrama puro no produce sugerencias si el contexto nunca se vio. Los sistemas reales usan **retroceso**: se piden candidatos al trigrama; si no alcanzan, se completa con el bigrama y, si aún faltan, con el unigrama.

Implemente `ranking(ctx, m)`: lista de **hasta `m` palabras** (cadenas), ordenadas así: primero las del trigrama (de más a menos frecuentes), luego las del bigrama que no estén ya, luego las del unigrama. Nunca incluya `<unk>` (índice 0). `ctx` es la lista de los últimos 3 índices.

In [8]:
def ranking(ctx, m=60):
    out = []
    for n in (3, 2, 1, 0):
        contexto = tuple(ctx[-n:]) if n else ()
        for w, _ in T[n].get(contexto, Counter()).most_common(m):
            if w != 0 and w not in out:
                out.append(w)
                if len(out) >= m:
                    return [vocab[i] for i in out[:m]]
    return [vocab[i] for i in out[:m]]


def topk(secuencia, k):
    aciertos = total = 0
    for i in range(3, len(secuencia)):
        if secuencia[i] == 0:
            continue
        total += 1
        aciertos += vocab[secuencia[i]] in ranking(secuencia[i - 3:i], k)
    return aciertos / total


contexto = [a_id["en"], a_id["un"], a_id["lugar"]]
print("después de 'en un lugar':", ranking(contexto, 3))
print(f"validación  top-1: {topk(val, 1):.3f} | top-3: {topk(val, 3):.3f}")

después de 'en un lugar': ['de', 'que', 'cerca']


validación  top-1: 0.155 | top-3: 0.278


In [9]:
top1_ng, top3_ng = topk(val, 1), topk(val, 3)
verificar("ranking", [a_id[w] for w in ranking(contexto, 3)] + [top1_ng, top3_ng]);

OK       ranking


**Pregunta de análisis:** (a) ¿Por qué `top-3` es bastante mayor que `top-1`? (b) Ejecute `ranking([a_id["xx"] if "xx" in a_id else 0, 0, 0], 3)`: ¿qué devuelve cuando no hay contexto útil y por qué es razonable?

Top-3 aumenta porque se acepta cualquiera de tres alternativas, mientras que top-1 solo acepta la primera. Se pasó de 15.54 % a 27.77 %, una diferencia de 12.23 puntos porcentuales.
La llamada con tres ceros devuelve y, a y de. Aunque se consulta un contexto sin palabras reconocidas, existen conteos de secuencias con <unk>; si faltan candidatos, se retrocede hasta las palabras frecuentes. El resultado es razonable como alternativa general, pero no expresa una intención concreta.


---
## Bloque 4 — Medir como un usuario: pulsaciones ahorradas (15 puntos)

Un acierto *top-3* cercano a 28 % no refleja la utilidad real, porque el usuario también **teclea las primeras letras** y entonces las sugerencias se filtran por prefijo. Se simula así: para escribir la palabra `w`, después de teclear $j$ letras ($j=0,1,2,\dots$) el sistema muestra las 3 primeras palabras de `ranking` que empiecen por esas $j$ letras. Si `w` está entre ellas, el usuario toca la sugerencia: costo $= j+1$ pulsaciones. Si nunca aparece, teclea la palabra completa: costo $=$ `len(w)`.

El costo nunca se considera mayor que `len(w)`.

In [10]:
def costo(palabra, rk):
    for j in range(len(palabra) + 1):
        sugerencias = list(itertools.islice((w for w in rk if w.startswith(palabra[:j])), 3))
        if palabra in sugerencias:
            return min(j + 1, len(palabra))
    return len(palabra)



def ahorro(fn_ranking, n_palabras=3000):
    """Fracción de pulsaciones ahorradas sobre las primeras n_palabras de validación."""
    gasto = largo = 0
    for i in range(3, 3 + n_palabras):
        if val[i] == 0:
            continue
        w = vocab[val[i]]
        gasto += costo(w, fn_ranking(val[i - 3:i]))
        largo += len(w)
    return 1 - gasto / largo


print("costo('quijote', ...) tras 'don':", costo("quijote", ranking([0, 0, a_id["don"]])))
ahorro_ng = ahorro(ranking)
ahorro_uni = ahorro(lambda ctx: [vocab[i] for i, _ in T[0][()].most_common(60) if i != 0])
print(f"pulsaciones ahorradas: n-gramas con retroceso = {ahorro_ng:.3f} | solo unigramas = {ahorro_uni:.3f}")

costo('quijote', ...) tras 'don': 1


pulsaciones ahorradas: n-gramas con retroceso = 0.301 | solo unigramas = 0.129


In [11]:
verificar("ahorro", [costo("quijote", ["quijote", "don"]), costo("don", ["que", "de"]), costo("sancho", ["sancho"] ), ahorro_ng, ahorro_uni]);

OK       ahorro


**Pregunta de análisis:** (a) El modelo de unigramas **no mira el contexto** y aun así ahorra pulsaciones. ¿Por qué? (b) ¿Cuántos puntos de ahorro aporta el contexto (n-gramas con retroceso frente a unigramas)? Valore si esa mejora justifica la complejidad adicional.

Los unigramas ahorran pulsaciones porque las palabras frecuentes se pueden seleccionar directamente o después de escribir un prefijo, sin conocer el contexto. Su ahorro fue de 12.89 %.
Con retroceso se alcanzó 30.12 %, una mejora de 17.23 puntos porcentuales. En esta simulación, esa diferencia justifica almacenar más conteos porque se evita aproximadamente una sexta parte adicional de las pulsaciones; en un teclado real también se tendrían que medir memoria, demora y esfuerzo de selección.


---
## Bloque 5 — Una red neuronal que predice la siguiente palabra (10 puntos)

Es la idea de Bengio (2003): cada una de las 3 palabras previas se convierte en un *embedding*, se concatenan, pasan por una capa oculta y una capa final produce un puntaje para cada palabra del vocabulario.

`Embedding(V, 32)` → concatenar 3 → `Linear(96, 128)` → `Tanh` → `Linear(128, V)`.

In [12]:
class RedBengio(nn.Module):
    def __init__(self, n_vocab, dim=32, oculto=128, contexto=3):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, dim)
        self.mlp = nn.Sequential(nn.Linear(contexto * dim, oculto), nn.Tanh(), nn.Linear(oculto, n_vocab))

    def forward(self, x):
        return self.mlp(self.emb(x).flatten(start_dim=1))


modelo = RedBengio(V)
n_params = sum(p.numel() for p in modelo.parameters())
print("parámetros:", n_params)

parámetros: 817416


In [13]:
verificar("n_params", [n_params]);

OK       n_params


El entrenamiento ya está escrito. Ejecútelo (menos de 1 minuto en CPU).

In [14]:
def entrenar_red(secuencia, epocas=6, semilla=0):
    X = torch.tensor([secuencia[i:i + 3] for i in range(len(secuencia) - 3)])
    Y = torch.tensor(secuencia[3:])
    torch.manual_seed(semilla)
    red = RedBengio(V)
    opt = torch.optim.Adam(red.parameters(), lr=3e-3)
    for ep in range(epocas):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 512):
            ix = perm[i:i + 512]
            perdida = nn.functional.cross_entropy(red(X[ix]), Y[ix])
            opt.zero_grad()
            perdida.backward()
            opt.step()
        print(f"época {ep + 1} | pérdida (último lote) {perdida.item():.3f}")
    return red.eval()


modelo = entrenar_red(ent)


@torch.no_grad()
def ranking_red(ctx, m=60, red=None):
    red = red or modelo
    logits = red(torch.tensor([list(ctx)]))[0].clone()
    logits[0] = -float("inf")                     # nunca sugerir <unk>
    return [vocab[i] for i in logits.topk(m).indices.tolist()]


def topk_red(k):
    aciertos = total = 0
    for i in range(3, len(val)):
        if val[i] == 0:
            continue
        total += 1
        aciertos += vocab[val[i]] in ranking_red(val[i - 3:i], k)
    return aciertos / total


top1_red, top3_red = topk_red(1), topk_red(3)
ahorro_red = ahorro(ranking_red)
print(f"red neuronal: top-1 {top1_red:.3f} | top-3 {top3_red:.3f} | ahorro {ahorro_red:.3f}")
verificar("red_ok", [float(top3_red > 0.15 and ahorro_red > 0.2)]);

época 1 | pérdida (último lote) 5.052


época 2 | pérdida (último lote) 5.115


época 3 | pérdida (último lote) 4.617


época 4 | pérdida (último lote) 4.401


época 5 | pérdida (último lote) 4.256


época 6 | pérdida (último lote) 4.255


red neuronal: top-1 0.142 | top-3 0.267 | ahorro 0.281
OK       red_ok


**Pregunta de análisis:** (a) ¿La red le ganó al n-grama en `top-3` y en `ahorro`? (b) La red tiene aproximadamente 800,000 parámetros y el corpus solo 380,000 palabras. ¿Qué riesgo implica eso y qué esperaría con 100 veces más texto y por qué?

La red no superó a los n-gramas: obtuvo 26.7 % en top-3 y 28.1 % de ahorro, frente a 27.8 % y 30.1 %. La diferencia fue de aproximadamente 1.1 y 2.0 puntos porcentuales, respectivamente.
Con 817,416 parámetros y 345,273 palabras de entrenamiento existe riesgo de memorizar patrones y generalizar poco. Con cien veces más texto del mismo dominio se esperaría aprender representaciones más útiles y mejorar las predicciones, aunque también mejorarían los conteos; una victoria de la red tendría que comprobarse.


---
## Bloque 6 — Reto de investigación: ¿quién gana, y en qué? (30 puntos)

Este bloque no contiene `TODO` obligatorios de código: plantea una pregunta que usted debe responder con **evidencia**.
Se califica el diseño y la interpretación, no que el número salga bonito.

### Parte A — La tabla (10 puntos)

Ejecute la comparación de los modelos y **agregue una columna propia** que le parezca informativa
(por ejemplo: pulsaciones ahorradas solo en palabras de más de 6 letras; o ahorro solo cuando el trigrama conoce el contexto; o tiempo por predicción).

In [15]:
def ahorro_largas(fn_ranking, n_palabras=3000):
    gasto = largo = cantidad = 0
    for i in range(3, 3 + n_palabras):
        if val[i] == 0 or len(vocab[val[i]]) <= 6:
            continue
        w = vocab[val[i]]
        gasto += costo(w, fn_ranking(val[i - 3:i]))
        largo += len(w)
        cantidad += 1
    return 1 - gasto / largo, cantidad

ranking_uni = lambda ctx: [vocab[i] for i, _ in T[0][()].most_common(60) if i != 0]
largas_uni, cantidad_largas = ahorro_largas(ranking_uni)
largas_ng, _ = ahorro_largas(ranking)
largas_red, _ = ahorro_largas(ranking_red)
filas = [
    ("unigramas", None, None, ahorro_uni),
    ("n-gramas con retroceso", top1_ng, top3_ng, ahorro_ng),
    ("red neuronal", top1_red, top3_red, ahorro_red),
]
columna_largas = dict(zip([fila[0] for fila in filas], [largas_uni, largas_ng, largas_red]))
print(f"{'modelo':<26}{'top-1':>8}{'top-3':>8}{'ahorro':>9}{'ahorro >6 letras':>18}")
for nom, a, b, c in filas:
    print(f"{nom:<26}{(f'{a:.3f}' if a else '--'):>8}{(f'{b:.3f}' if b else '--'):>8}{c:>9.3f}{columna_largas[nom]:>18.3f}")
print("Palabras evaluadas en la columna propia:", cantidad_largas)


modelo                       top-1   top-3   ahorro  ahorro >6 letras
unigramas                       --      --    0.129             0.091
n-gramas con retroceso       0.155   0.278    0.301             0.310
red neuronal                 0.142   0.267    0.281             0.286
Palabras evaluadas en la columna propia: 474


*Interprete en 3 a 4 líneas:* ¿qué modelo ganó en cada columna y cuánto? ¿Es la diferencia lo bastante grande como para que un usuario la note? Explique su columna propia y qué le enseñó.

Los n-gramas ganaron en top-1, top-3 y ahorro: 15.5 %, 27.8 % y 30.1 %, frente a 14.2 %, 26.7 % y 28.1 % de la red.
La ventaja de ahorro frente a unigramas fue de 17.2 puntos; frente a la red fue de unos 2.0 puntos, una diferencia más pequeña cuya percepción requiere una prueba con usuarios.
La columna propia mide el ahorro en 474 palabras conocidas de más de seis letras: 31.0 % para n-gramas, 28.6 % para la red y 9.1 % para unigramas.
Se observó que el contexto también ayuda en palabras largas. Las métricas de esta tabla excluyen <unk>, por lo que no representan el ahorro sobre todas las palabras escritas.


### Parte B — Su propio experimento (20 puntos)

Elija **una** de estas preguntas, formule una hipótesis y póngala a prueba:

1. **Curva de aprendizaje.** Entrene el trigrama y la red con el 10 %, 30 % y 100 % del corpus de entrenamiento. ¿Con cuánto texto la red alcanza o supera al n-grama? (Bengio promete que la red gana cuando hay datos; ¿se cumple aquí?)
2. **Cambio de dominio.** Escriba usted 40 a 60 palabras de mensajes cotidianos (del tipo que enviaría a un compañero de clase) y mida las sugerencias sobre **su** texto. ¿Cuánto cae el ahorro frente a validación? ¿Qué palabras de su texto son `<unk>`? Proponga cómo adaptaría el modelo a su estilo.
3. **Contexto y vocabulario.** Mida el ahorro con `n` = 1, 2, 3 (sin retroceso al unigrama) y con vocabularios de 1,000, 5,000 y 15,000 palabras. ¿Qué pesa más: más contexto o más vocabulario?

Entregue, en la celda de código y la de texto siguientes:

- **Hipótesis** (1 línea): qué espera y por qué.
- **Evidencia**: una tabla o cifra producida por su código.
- **Conclusión** (máx. 5 líneas), incluyendo **una limitación** de su experimento.

In [16]:
resultados_experimento = []
limite = int(len(tokens) * 0.9)
texto_ent, texto_val = tokens[:limite], tokens[limite:]
muestra = list(range(3, min(3003, len(texto_val))))
print("Sin retroceso; 60 candidatos y hasta 3 sugerencias por prefijo.")
print(f"{'vocabulario':>12}{'contexto':>10}{'cobertura':>12}{'ahorro total':>15}{'ahorro conocidas':>19}")
for tam in (1000, 5000, 15000):
    vocab_exp = ["<unk>"] + [w for w, _ in Counter(texto_ent).most_common(tam - 1)]
    ids_exp = {w: i for i, w in enumerate(vocab_exp)}
    ent_exp = [ids_exp.get(w, 0) for w in texto_ent]
    val_exp = [ids_exp.get(w, 0) for w in texto_val]
    for contexto_exp in (1, 2, 3):
        tabla_exp = entrenar(ent_exp, contexto_exp)
        gasto_total = largo_total = gasto_conocidas = largo_conocidas = conocidas = 0
        for i in muestra:
            palabra = texto_val[i]
            ctx_exp = tuple(val_exp[i - contexto_exp:i])
            candidatos = [vocab_exp[w] for w, _ in tabla_exp.get(ctx_exp, Counter()).most_common() if w != 0][:60]
            gasto = costo(palabra, candidatos)
            gasto_total += gasto
            largo_total += len(palabra)
            if val_exp[i] != 0:
                conocidas += 1
                gasto_conocidas += gasto
                largo_conocidas += len(palabra)
        cobertura = conocidas / len(muestra)
        ahorro_total = 1 - gasto_total / largo_total
        ahorro_conocidas = 1 - gasto_conocidas / largo_conocidas
        resultados_experimento.append((tam, contexto_exp, cobertura, ahorro_total, ahorro_conocidas))
        print(f"{tam:>12}{contexto_exp:>10}{cobertura:>12.3f}{ahorro_total:>15.3f}{ahorro_conocidas:>19.3f}")


Sin retroceso; 60 candidatos y hasta 3 sugerencias por prefijo.
 vocabulario  contexto   cobertura   ahorro total   ahorro conocidas


        1000         1       0.790          0.222              0.338


        1000         2       0.790          0.188              0.286


        1000         3       0.790          0.108              0.165


        5000         1       0.914          0.229              0.270


        5000         2       0.914          0.174              0.205


        5000         3       0.914          0.078              0.092


       15000         1       0.954          0.229              0.250


       15000         2       0.954          0.161              0.176


       15000         3       0.954          0.067              0.073


Hipótesis: se esperaba que aumentar el vocabulario mejorara la cobertura y que los contextos largos redujeran el ahorro cuando no hubiera retroceso, debido a la escasez de ejemplos.

Evidencia: se evaluaron las mismas 3,000 palabras de validación con nueve combinaciones. Los vocabularios se construyeron solo con entrenamiento y las palabras desconocidas se contaron con su costo completo. La cobertura fue de 79.0 %, 91.4 % y 95.4 % con 1,000, 5,000 y 15,000 palabras. Con contexto de una palabra, el ahorro total fue 22.2 %, 22.9 % y 22.9 %; con tres palabras fue 10.8 %, 7.8 % y 6.7 %.

Conclusión: se respaldó la hipótesis sobre cobertura y escasez de contextos. Aumentar el vocabulario casi no mejoró el ahorro total con contexto corto, mientras que pasar de una a tres palabras de contexto lo redujo entre 11.4 y 16.3 puntos. En estas condiciones pesó más la longitud del contexto: los contextos largos necesitaron más ejemplos. Como limitación, se usó una sola muestra de una novela y no se aplicó retroceso; el resultado no demuestra que los contextos cortos sean mejores en otros corpus o modelos.


---
## Bloque 7 — Análisis y conclusión (5 puntos)

2 a 4 líneas cada una.

1. El modelo se entrenó con una novela del siglo XVII. Si se entrenara con todo lo que un usuario escribe en su celular, ¿qué cambiaría en la calidad de las sugerencias y qué riesgos de privacidad aparecerían (contraseñas, números de tarjeta, mensajes personales)?
2. Identifique la sugerencia del modelo que le pareció más convincente y la medición de este laboratorio que aumentó o redujo su confianza en ella. Cite el resultado.

### Sus respuestas — Bloque 7

1. Al entrenar con mensajes personales se esperaría sugerir mejor nombres, expresiones y temas frecuentes del usuario, aunque los errores y cambios de estilo también influirían. Se podrían almacenar o reproducir datos privados, como contraseñas, tarjetas y mensajes. Se deberían excluir campos sensibles, reducir la retención y permitir borrar los datos; el aprendizaje local ayudaría a limitar su exposición.

2. La sugerencia quijote después de don resultó convincente: se observaron 1,863 continuaciones de ese tipo en entrenamiento y se necesitó una pulsación para seleccionarla. Sin embargo, el top-3 de los n-gramas fue de 27.77 % en validación, por lo que un ejemplo conocido no garantiza acertar en otros contextos. El ahorro de 30.12 % mostró utilidad parcial, incluso sin acertar siempre desde el inicio.
